In [ ]:
import pandas as pd
import numpy as np
import random
from datetime import timedelta

In [ ]:
class SupplyChainChaosEngine:
    """
    Simulates 17 specific systemic and operational failures in a 
    D2C supply chain dataset to test pipeline resilience.
    """
    def __init__(self, df_inventory, df_nodes):
        self.df = df_inventory.copy()
        self.df_nodes = df_nodes.copy()
        self.df['timestamp'] = pd.to_datetime(self.df['timestamp'])

    # --- CATEGORY 1: TEMPORAL CHAOS (Time issues) ---
    def apply_temporal_chaos(self):
        # 1. Z-Time Trap: Shift 10% of rows by 5.5 hours (UTC vs IST drift)
        idx1 = self.df.sample(frac=0.10).index
        self.df.loc[idx1, 'timestamp'] += timedelta(hours=5, minutes=30)
        
        # 3. Midnight Rollover: Truncate 5% of timestamps to Date-only (00:00:00)
        idx2 = self.df.sample(frac=0.05).index
        self.df.loc[idx2, 'timestamp'] = self.df.loc[idx2, 'timestamp'].dt.normalize()
        
        # 5. Batch Heartbeat: Force 80% of data for March 15th to the exact same second
        target_date = '2026-03-15'
        mask = self.df['timestamp'].dt.date == pd.to_datetime(target_date).date()
        self.df.loc[mask, 'timestamp'] = pd.Timestamp(f"{target_date} 23:59:59")
        
        # Latency Smear: Massive lag on 2% of rows
        self.df.loc[self.df.sample(frac=0.02).index, 'processing_lag_hours'] = 155.5
        return self

    # --- CATEGORY 2: SCHEMA & STRUCTURAL CHAOS (Formatting issues) ---
    def apply_structural_chaos(self):
        # 2. Schema Poisoning: Inject " units" string into numeric units_sold
        idx = self.df.sample(n=50).index
        self.df.loc[idx, 'units_sold'] = self.df.loc[idx, 'units_sold'].astype(str) + " units"
        
        # Address Emoji/Special Char Corruption in Nodes Table
        bad_chars = ["🚚 Hub-Alpha", "Sector 12 || 🏠", "Industrial Area\tPhase-1", "MOHALI-999-###"]
        for _ in range(3):
            self.df_nodes.loc[self.df_nodes.sample(1).index, 'city'] = random.choice(bad_chars)
            
        # 5. Null Value Silent Decay: Drop reasons even when units_rto > 0
        rto_mask = (self.df['units_rto'] > 0)
        null_idx = self.df[rto_mask].sample(n=20).index
        self.df.loc[null_idx, 'rto_reason'] = np.nan
        return self

    # --- CATEGORY 3: OPERATIONAL & LOGIC CHAOS (Business failures) ---
    def apply_logic_chaos(self):
        # 1. Ghost Inventory: Make units_sold > stock_on_hand
        ghost_idx = self.df.sample(n=20).index
        self.df.loc[ghost_idx, 'units_sold'] = self.df.loc[ghost_idx, 'stock_on_hand'] + 500
        
        # 2. Duplicate Transaction Echoes: Double-reporting
        dupes = self.df.sample(n=100)
        self.df = pd.concat([self.df, dupes], ignore_index=True)
        
        # 3. Node Blackout: Delete 4 days of data for one specific node
        self.df = self.df[~((self.df['node_id'] == 'NODE_006') & 
                           (self.df['timestamp'].dt.day.isin([10, 11, 12, 13])))]
        
        # Reverse Logistics Void: High RTO but stock doesn't reflect the return
        void_idx = self.df.sample(n=15).index
        self.df.loc[void_idx, 'units_rto'] = 100
        self.df.loc[void_idx, 'stock_on_hand'] = 2 # Stock stays low
        return self

    # --- CATEGORY 4: INTEGRITY & MARKET CHAOS ---
    def apply_market_chaos(self):
        # 3. GST-Inclusive Math Drift: Multiply Revenue by 1.18 randomly
        if 'revenue' in self.df.columns:
            idx = self.df.sample(frac=0.15).index
            self.df.loc[idx, 'revenue'] = self.df.loc[idx, 'revenue'] * 1.18
            
        # 4. SKU Migration / Identity Crisis
        mask = (self.df['product_id'] == 'PROD_001') & (self.df['timestamp'].dt.day < 10)
        self.df.loc[mask, 'product_id'] = 'PROD_001_OLD_SKU'
        
        # Fat Finger Outliers
        self.df.loc[self.df.sample(n=2).index, 'units_sold'] = 999999
        return self

    def save_chaos(self):
        self.df.to_csv('03_inventory_chaotic.csv', index=False)
        self.df_nodes.to_csv('03_nodes_chaotic.csv', index=False)
        print("Data Armageddon complete. 17 chaos types successfully injected.")

# --- EXECUTION ---
# Load baseline (Assume they are in your path)
df_inv = pd.read_csv('02_inventory_test_baseline.csv')
df_nod = pd.read_csv('01_nodes_test.csv')

# Run Engine
chaos = SupplyChainChaosEngine(df_inv, df_nod)
(chaos.apply_temporal_chaos()
     .apply_structural_chaos()
     .apply_logic_chaos()
     .apply_market_chaos()
     .save_chaos())